In [8]:
# Capstone Project 3 
# Smart Marketing Prediction System (ML Pipeline Project)
# Scenario
# A fast-growing e-commerce company called ShopEasy is struggling with inefficient marketing campaigns.

# Every day thousands of users visit their website. The marketing team spends a large amount of money showing ads, discounts, and promotional emails, but they don't know which customers are actually likely to buy something.

# Currently:

# Many customers browse but never purchase

# Marketing money is wasted on the wrong users

# The company wants to predict purchase probability

# The data science team has been asked to build a machine learning system that predicts whether a customer will purchase a product during a session.

# If the system predicts high probability of purchase, the system will:

# show personalized product recommendations

# offer targeted discounts

# prioritize marketing campaigns

# If the system predicts low probability, the company will avoid spending marketing resources.

# However, the dataset contains both numerical and categorical features, so the data science team must design a complete ML pipeline.

# Dataset is available in DatasetCapstoneProject3 in the github repo link https://github.com/himanshusar123/Datasets

# Business Objective
# Build a machine learning model that predicts whether a user will purchase (1) or not purchase (0) during a website session.

# The model must be implemented using scikit-learn pipelines, including:

# Encoding techniques

# Feature preprocessing

# Model training

# Model selection

# Hyperparameter tuning





import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

df=pd.read_excel("/Users/ayush/Desktop/capgemini_assignments/capstone_projects/DatasetCapstoneProject3.xlsx")
X = df[['Age',	'Gender','Device','Traffic_Source','Time_on_Website',	'Pages_Visited',	'Ad_Clicks','Previous_Purchases']]
y = df['Purchased']

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# defining features:-
numeric_features = ['Age','Time_on_Website','Pages_Visited','Ad_Clicks','Previous_Purchases']
categorical_features = ['Gender','Device','Traffic_Source']

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# applying preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)


param_grid = {
#  classifier__C means:
# "Change the C hyperparameter of the Logistic Regression model inside the pipeline step named classifier."
    "classifier__C": [0.01, 0.1, 1, 10],
    # It means 4 different versions of Logistic Regression trained with different regularization strengths
    "classifier__penalty": ["l2"],  # l1 requires solver='liblinear'
    
    "classifier__solver": ["lbfgs", "saga"]
    # So total combinations:
    # 4 C values × 2 solvers = 8 models
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3, 
    # cross validation
    # Changed from 5 to 3 to accommodate the minority class size
# #  Then with cv=3:
# Each model is trained 3 times (3 folds)
# So total training runs:
# 8 × 3 = 24 training runs
# 8 from param grids look uppper part
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

accuracy = accuracy_score(y_test, y_pred)
print("Best Parameters:", grid_search.best_params_)
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))

# now checking for new user
new_user = pd.DataFrame({
    "Age": [30],
    "Gender": ["Male"],
    "Device": ["Mobile"],
    "Traffic_Source": ["Social Media"],
    "Time_on_Website": [12],
    "Pages_Visited": [5],
    "Ad_Clicks": [2],
    "Previous_Purchases": [1]
})
# checking prediction for new users
prediction = grid_search.predict(new_user)
print("Prediction:", prediction)






Best Parameters: {'classifier__C': 1, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3

Prediction: [0]
